In [9]:
import warnings
# Ignore all warnings
warnings.filterwarnings("ignore")

import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn import metrics
from sklearn.model_selection import train_test_split

import category_encoders as ce
from sklearn.preprocessing import OneHotEncoder
from category_encoders import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree

from sklearn.model_selection import cross_val_score, StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score, precision_score, classification_report, f1_score, fbeta_score, make_scorer

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC


In [10]:
df = pd.read_csv(r'C:\Users\ss\OneDrive\Desktop\CUSTOMER PREDICTION\Data\preprocesed_data.csv')
df = pd.DataFrame(df)
df.head()

,Year_Birth,Education,Marital_Status,Income,Recency,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Response
0,1957,Graduation,Single,58138.0,58,0,0,0,0,0,0,1
1,1954,Graduation,Single,46344.0,38,0,0,0,0,0,0,0
2,1965,Graduation,Together,71613.0,26,0,0,0,0,0,0,0
3,1984,Graduation,Together,26646.0,26,0,0,0,0,0,0,0
4,1981,PhD,Married,58293.0,94,0,0,0,0,0,0,0


In [11]:
df.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2213 entries, 0 to 2212
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Year_Birth      2213 non-null   int64  
 1   Education       2213 non-null   object 
 2   Marital_Status  2213 non-null   object 
 3   Income          2213 non-null   float64
 4   Recency         2213 non-null   int64  
 5   AcceptedCmp3    2213 non-null   int64  
 6   AcceptedCmp4    2213 non-null   int64  
 7   AcceptedCmp5    2213 non-null   int64  
 8   AcceptedCmp1    2213 non-null   int64  
 9   AcceptedCmp2    2213 non-null   int64  
 10  Complain        2213 non-null   int64  
 11  Response        2213 non-null   int64  
dtypes: float64(1), int64(9), object(2)
memory usage: 207.6+ KB


**Feature Engineering**

The columns **education** and **marital status** are categorical data, so they need to be converted into numerical form to be processed by the model.

* The **education** column is processed using **ordinal encoding**, as it has an inherent hierarchical order.
* The **marital status** column is processed using **one-hot encoding**, since each category is independent and has no ordinal relationship.


In [12]:
mapping = [{
    'col': 'Education',
    'mapping': {
        None: 0,
        'Basic': 1,
        '2n Cycle': 2,
        'Graduation': 3,
        'Master': 4,
        'PhD': 5
    }
}]

ct = ColumnTransformer(
    transformers=[
        ('onehot',
         OneHotEncoder(drop='first', handle_unknown='ignore'),
         ['Marital_Status']),

        ('ordinal',
         ce.OrdinalEncoder(
             mapping=mapping,
             handle_unknown='value',
             handle_missing='value'
         ),
         ['Education'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

In [13]:
dft = ct.fit_transform(df)

dft = pd.DataFrame(
    dft,
    columns=ct.get_feature_names_out(),
    index=df.index
)

dft.head()

,Marital_Status_Married,Marital_Status_Single,Marital_Status_Together,Marital_Status_Widow,Education,Year_Birth,Income,Recency,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Response
0,0.0,1.0,0.0,0.0,3.0,1957.0,58138.0,58.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0.0,1.0,0.0,0.0,3.0,1954.0,46344.0,38.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,1.0,0.0,3.0,1965.0,71613.0,26.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,1.0,0.0,3.0,1984.0,26646.0,26.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,5.0,1981.0,58293.0,94.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


All columns in the dataframe have been converted to floats after passing through the column transformer pipeline

In [15]:
dft = dft.astype('Int64')
dft['Income'] = dft['Income'].astype(float)
df = dft

In [16]:
def dataframe_observation(df):
    unique_value = df.nunique()
    data_type  = df.dtypes
    sample_unique = [df[col].unique() for col in df.columns]

    df = pd.DataFrame({
        'Total Unique Value' : unique_value,
        'Data Type' : data_type,
        'Sample Of value' : sample_unique
    })

    return df

dataframe_observation(df)

,Total Unique Value,Data Type,Sample Of value
Marital_Status_Married,2,Int64,"[0, 1]"
Marital_Status_Single,2,Int64,"[1, 0]"
Marital_Status_Together,2,Int64,"[0, 1]"
Marital_Status_Widow,2,Int64,"[0, 1]"
Education,5,Int64,"[3, 5, 4, 1, 2]"
Year_Birth,56,Int64,"[1957, 1954, 1965, 1984, 1981, 1967, 1971, 198..."
Income,1971,float64,"[58138.0, 46344.0, 71613.0, 26646.0, 58293.0, ..."
Recency,100,Int64,"[58, 38, 26, 94, 16, 34, 32, 19, 68, 59, 82, 5..."
AcceptedCmp3,2,Int64,"[0, 1]"
AcceptedCmp4,2,Int64,"[0, 1]"


**Target Column Distribution**

In [17]:
df['Response'].value_counts()/len(df)*100

Response
0    84.952553
1    15.047447
Name: count, dtype: Float64

The distribution of the target variable indicates a class imbalance, where only about 15% of customers accepted the campaign, while the remaining 85% did not respond positively.

### **Modeling**


**Define Feature and Target**

In [18]:
# define features and target

X = df.drop(columns='Response')
y = df['Response']

Split Data into Training and Testing datasets

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

**Model Training**

SVC and Logistic regression are sensitive to feature scales so I standerdize them to enable them learn more appropriately and efficiently

In [20]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000))
    ]),

    'SVC': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC())
    ]),

    'Decision Tree': DecisionTreeClassifier(
        random_state=12, max_depth=5
    ),

    'Random Forest': RandomForestClassifier(
        random_state=12
    )
}

In [21]:
for name, model in models.items():
    model.fit(X_train, y_train)

**Evaluation**